<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

repo_root = None
for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    scripts = p / "work" / "scripts"
    if (scripts / "warehouse_frame.py").exists():
        sys.path.insert(0, str(scripts))
        repo_root = p
        break
else:
    raise FileNotFoundError("work/scripts/warehouse_frame.py not found")

from warehouse_frame import load_notebook_frame

df = load_notebook_frame()
print(f"Outputs folder: {repo_root / 'work' / 'outputs'}")

Hugging Face warehouse: 79,576 pages, 26 clients
Features: Jan-Feb 2026. Label: Apr impressions < 80% of Mar.
Declining rate: 0.557
Outputs folder: C:\Users\rimla\Downloads\flyrank-ml-assignments-main\flyrank-ml-assignments-main\work\outputs


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes
The queue is ranked by the Random Forest score. Each page is scored out-of-fold (the model for that page was not trained on that client). Labels are **not** used to assign the action.

**Action categories:**
- **CONTENT_REFRESH_PRIORITY** (MODEL_RISK_HIGH_VISIBILITY): high model score and visible in search
- **CONTENT_REFRESH_MODERATE** (STALE_VISIBLE): old page that still gets impressions, not already priority
- **MONITOR_STABLE** (MONITOR): not high risk and not the stale-visible case
- **REVIEW_THRESHOLD** (LOW_VISIBILITY): almost no impressions

**Rules (no `trend_direction`):**
- **MODEL_RISK_HIGH_VISIBILITY**: model score in the top 30% AND impressions_90d ≥ 100 AND 0 < avg_position ≤ 20
- **STALE_VISIBLE**: days_since_last_update ≥ 180 AND impressions_90d ≥ 500
- **LOW_VISIBILITY**: impressions_90d < 50
- **MONITOR**: everything else

Cohort sizes are printed by the next cell. Do not hard-code percents. On this warehouse run only CONTENT_REFRESH_PRIORITY and MONITOR_STABLE appeared. STALE_VISIBLE and LOW_VISIBILITY matched no remaining pages (most pages already have impressions well above 50).

In [14]:
features = ["content_age_days", "days_since_last_update", "impressions_90d", "ctr", "avg_position"]
X = df[features].fillna(0)
y = df["is_declining"]
groups = df["client_id"]

# Out-of-fold scores: every page is scored by a model that did not train on that client
oof = np.zeros(len(df), dtype=float)
gkf = GroupKFold(n_splits=5)
for train_idx, test_idx in gkf.split(X, y, groups):
    rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof[test_idx] = rf.predict_proba(X.iloc[test_idx])[:, 1]

df["model_score"] = oof

high_risk = df["model_score"] >= df["model_score"].quantile(0.70)
visible = (df["impressions_90d"] >= 100) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
low_vis = df["impressions_90d"] < 50

conditions = [
    high_risk & visible,
    stale_visible,
    low_vis,
]
choices_action = ["CONTENT_REFRESH_PRIORITY", "CONTENT_REFRESH_MODERATE", "REVIEW_THRESHOLD"]
choices_reason = ["MODEL_RISK_HIGH_VISIBILITY", "STALE_VISIBLE", "LOW_VISIBILITY"]

df["action_label"] = np.select(conditions, choices_action, default="MONITOR_STABLE")
df["reason_code"] = np.select(conditions, choices_reason, default="MONITOR")

priority_scores = {
    "CONTENT_REFRESH_PRIORITY": 100,
    "CONTENT_REFRESH_MODERATE": 60,
    "MONITOR_STABLE": 10,
    "REVIEW_THRESHOLD": 5,
}
df["priority_score"] = df["action_label"].map(priority_scores)
df["final_score"] = df["priority_score"] + df["model_score"] * 10

final_ranked_queue = df.sort_values(by=["final_score", "model_score"], ascending=False)

print("=== ACTION PLAYBOOK COHORTS ===")
print(final_ranked_queue["action_label"].value_counts().to_string())
print(final_ranked_queue["action_label"].value_counts(normalize=True).mul(100).round(1).astype(str).add("%").to_string())
print(f"\nTotal pages: {len(final_ranked_queue):,}")

print("\n=== TOP 10 ===")
cols = ["action_label", "reason_code", "model_score", "impressions_90d", "avg_position"]
print(final_ranked_queue[cols].head(10).to_string(index=False))

=== ACTION PLAYBOOK COHORTS ===
action_label
MONITOR_STABLE              60877
CONTENT_REFRESH_PRIORITY    18699
action_label
MONITOR_STABLE              76.5%
CONTENT_REFRESH_PRIORITY    23.5%

Total pages: 79,576

=== TOP 10 ===
            action_label                reason_code  model_score  impressions_90d  avg_position
CONTENT_REFRESH_PRIORITY MODEL_RISK_HIGH_VISIBILITY     0.933801            111.0     16.210526
CONTENT_REFRESH_PRIORITY MODEL_RISK_HIGH_VISIBILITY     0.931639            224.0     19.298165
CONTENT_REFRESH_PRIORITY MODEL_RISK_HIGH_VISIBILITY     0.931639            252.0     19.863248
CONTENT_REFRESH_PRIORITY MODEL_RISK_HIGH_VISIBILITY     0.931639            100.0     19.197368
CONTENT_REFRESH_PRIORITY MODEL_RISK_HIGH_VISIBILITY     0.931600            133.0     19.752294
CONTENT_REFRESH_PRIORITY MODEL_RISK_HIGH_VISIBILITY     0.931408            186.0     19.082418
CONTENT_REFRESH_PRIORITY MODEL_RISK_HIGH_VISIBILITY     0.931408            181.0     19.616279
C

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use Framework and Boundary Limits

**Intended Domain Use**: This content action playbook is decision-support for content teams and SEO specialists. It ranks which pages to review first using Jan–Feb 2026 search signals and an Apr-vs-Mar drop label.

**Who Uses This**:
- Content editors and SEO specialists who decide where to invest editorial time
- Content strategists planning quarterly refresh calendars
- Site managers overseeing large content portfolios

**Known System Limits**:
- **No causal claims**: Ranking a page is not a claim that a rewrite will recover traffic
- **Algorithm shifts**: Cannot predict Google algorithm updates or ranking changes
- **External factors**: Seasonal trends, competitor actions, and market changes not captured
- **Future drop, not a rewrite outcome**: The label is Apr impressions < 80% of Mar, after Jan–Feb features. That is a next-month drop, not a same-window proxy, and still not proof a refresh will help
- **Single-channel data**: Only considers organic search, not other traffic sources
- **Forest vs rule**: On client-holdout fold 1 the fair rule (0.640 Precision@50) beats the forest (0.280). The playbook still uses out-of-fold forest scores to rank, with a person before anything ships

**Where It Stops Being Valid**:
- For brand new pages (< 90 days old) - insufficient history
- For pages with zero impressions - no measurable performance to assess
- During major search algorithm updates - historical patterns may not apply
- For highly seasonal content - normal seasonal fluctuations may be misidentified as decline

In [15]:
print(f"Action playbook boundaries defined for {len(final_ranked_queue):,} pages")
print(f"Primary users: Content teams, SEO specialists, content strategists")

Action playbook boundaries defined for 79,576 pages
Primary users: Content teams, SEO specialists, content strategists


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Guardrails and the Automation No-Go List

**Manual Review Requirements**:
Before taking action on flagged pages, content teams must manually verify:

1. **Seasonal pattern check**: Confirm decline isn't due to normal seasonal fluctuations (e.g., holiday content, annual events)
2. **Brand criticality assessment**: Ensure pages aren't core brand assets or homepage that require different treatment
3. **Technical audit**: Verify there are no technical SEO issues (crawl errors, indexing problems) causing the decline
4. **Content intent review**: Confirm the page's primary purpose and target audience are still relevant

**Automation No-Go List** (must always have human review):
- **Core brand pages**: Homepage, about pages, core product pages - never automate refresh decisions
- **Legal/compliance content**: Privacy policy, terms of service, disclaimers - require legal review
- **Financial/medical content**: YMYL (Your Money Your Life) pages - requires subject matter expert review
- **Active campaign landing pages**: Pages tied to current paid campaigns - may reflect temporary performance
- **New product launches**: Pages less than 90 days old - insufficient performance history

**Human Review Checklist**:
- [x] Confirm not a core brand page
- [x] Verify decline isn't seasonal/temporary
- [x] Check for technical SEO issues
- [x] Assess content relevance and accuracy
- [x] Consider business impact of changes

In [16]:
# Count pages that would require human review based on no-go criteria
# Estimate pages that might be core brand or critical content
# (In real implementation, this would use actual page type classification)

estimated_no_go_pages = final_ranked_queue[
    (final_ranked_queue['content_age_days'] < 90) |  # New pages
    (final_ranked_queue['impressions_90d'] > final_ranked_queue['impressions_90d'].quantile(0.95))  # Top visibility pages (likely core)
].shape[0]

print(f"Estimated pages requiring human review (no-go criteria): {estimated_no_go_pages:,}")
print(f"This represents ~{estimated_no_go_pages/len(final_ranked_queue)*100:.1f}% of total pages")
print("Human review guardrails protect critical brand and compliance content from automated decisions")

Estimated pages requiring human review (no-go criteria): 19,556
This represents ~24.6% of total pages
Human review guardrails protect critical brand and compliance content from automated decisions


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Operational Monitoring and Retraining Triggers

To ensure the action playbook remains effective over time, I need to monitor for data drift and performance degradation. The following triggers indicate when the model should be retrained or the playbook recalibrated:

**Data Drift Triggers**:
- **Feature distribution shift**: If the distribution of key features (impressions_90d, avg_position, ctr) shifts significantly from the baseline (±15% change in median values)
- **Target class imbalance**: If the declining rate changes dramatically (±10 percentage points from the baseline ~54%)
- **New content types**: If new content_type categories appear that weren't in the training data

**Performance Degradation Triggers**:
- **Precision@50 drop**: If the measured Precision@50 on new data falls below 80% of the baseline performance
- **False positive increase**: If the false positive rate increases above 30% (indicating too many stable pages flagged)
- **Reason code distribution shift**: If the distribution of reason codes changes significantly from the baseline

**Manual Review Triggers**:
- **Human feedback loop**: If content teams report that >25% of flagged pages don't actually need refresh
- **Business outcome tracking**: If refresh actions on top-priority pages don't show measurable improvement after 30 days

**Retraining Process**:
1. Collect new 90-day performance data
2. Re-run the entire pipeline with updated data
3. Re-validate using client-holdout split
4. Compare new model performance against baseline
5. Update action thresholds if performance gap narrows

In [17]:
out_dir = repo_root / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

baseline_metrics = {
    "baseline_impressions_median": float(df["impressions_90d"].median()),
    "baseline_avg_position_median": float(df["avg_position"].median()),
    "baseline_ctr_median": float(df["ctr"].median()),
    "baseline_declining_rate": float(df["is_declining"].mean()),
    "precision_50_threshold": 0.8,
    "drift_tolerance": 0.15,
    "fp_rate_threshold": 0.30,
}

print("=== MONITORING BASELINE METRICS ===")
for key, value in baseline_metrics.items():
    print(f"{key}: {value}")

metrics_path = out_dir / "action_playbook_baseline_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2)

print(f"\nWrote {metrics_path}")

=== MONITORING BASELINE METRICS ===
baseline_impressions_median: 1005.0
baseline_avg_position_median: 7.774551367425199
baseline_ctr_median: 0.13908205841446453
baseline_declining_rate: 0.5568764451593445
precision_50_threshold: 0.8
drift_tolerance: 0.15
fp_rate_threshold: 0.3

Wrote C:\Users\rimla\Downloads\flyrank-ml-assignments-main\flyrank-ml-assignments-main\work\outputs\action_playbook_baseline_metrics.json


### Cost / value vs random picking

Random picking matches the overall declining rate: 0.557. On client-holdout fold 1 the fair rule is 0.640 Precision@50 and the forest is 0.280 (test-fold base rate 0.439). Five-fold mean for the forest is 0.544.

If an editor spends about 4 hours on a page, the top 50 from the **fair rule** is 32 declining pages vs about 22 at the fold base rate. That is time allocation, not a claim that a rewrite will raise rank. Human review still has to catch seasonal pages, brand pages, and technical issues.

The 0.280 figure is Precision@50 for the forest on one held-out client. The fair rule (0.640) is the stronger shortlist on this warehouse label. The playbook still needs a person before anything ships, especially on brand, legal, and YMYL pages.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The playbook queue is written to `work/outputs/action_playbook_queue.csv`. That is separate from the Week-4 fair-rule file `baseline_action_score.csv`.


In [18]:
out_dir = repo_root / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

queue_cols = [
    "content_id",
    "client_id",
    "action_label",
    "reason_code",
    "model_score",
    "final_score",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
]
queue_path = out_dir / "action_playbook_queue.csv"
final_ranked_queue[queue_cols].to_csv(queue_path, index=False)
print(f"Wrote playbook queue: {queue_path}")
print(f"Rows: {len(final_ranked_queue):,}")
print(final_ranked_queue["action_label"].value_counts().to_string())


Wrote playbook queue: C:\Users\rimla\Downloads\flyrank-ml-assignments-main\flyrank-ml-assignments-main\work\outputs\action_playbook_queue.csv
Rows: 79,576
action_label
MONITOR_STABLE              60877
CONTENT_REFRESH_PRIORITY    18699


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.